In [ ]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import scipy.signal

import tensorflow as tf
import keras.backend as K

from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D,
    DepthwiseConv2D, BatchNormalization,
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import f1_score as sklearn_f1

try:
    import pywt
    _PYWT_AVAILABLE = True
except ImportError:
    _PYWT_AVAILABLE = False
    print('pywt not found — using hardcoded db4 coefficients.')

## Configuração

In [ ]:
DISPOSITIVOS = ['fridge']
IMAGENS      = ['tbg']

BATCH    = 32
N_RUNS   = 3
FOLDER_I = 'pickle_data'

# Variantes: nome -> (tipo de filtro, número de blocos DepthwiseConv2D substituídos)
# Substituição começa pelos PRIMEIROS blocos (camadas mais rasas), igual ao notebook 4.
VARIANTS = {
    'baseline':            ('none',    0),
    'fir_1block':          ('fir',     1),
    'fir_3blocks':         ('fir',     3),
    'wavelet_db4_1block':  ('wavelet', 1),
    'wavelet_db4_3blocks': ('wavelet', 3),
}

# Parâmetros FIR: filtro passa-baixas Hamming de 11 taps, corte em 0.4 * Nyquist
FIR_NUMTAPS = 11
FIR_CUTOFF  = 0.4
FIR_WINDOW  = 'hamming'

# Parâmetros Wavelet: Daubechies 4 (db4), subband LL (passa-baixas 2D)
WAVELET_NAME = 'db4'

## Camadas Customizadas: FIR e Daubechies

In [ ]:
class FIRDepthwise(tf.keras.layers.Layer):
    """
    Substitui DepthwiseConv2D por um filtro FIR 2D passa-baixas fixo (não treinável).

    O kernel 2D é o produto externo de dois filtros FIR 1D idênticos (separável):
        kernel_2d = np.outer(fir_1d, fir_1d)   → (numtaps, numtaps)

    O mesmo kernel é aplicado a todos os canais (depthwise, channel_multiplier=1).

    Entrada : (B, H, W, C)
    Saída   : (B, H/sh, W/sw, C)
    """

    def __init__(self, numtaps=FIR_NUMTAPS, cutoff=FIR_CUTOFF,
                 window=FIR_WINDOW, strides=(1, 1), **kwargs):
        super().__init__(**kwargs)
        self.numtaps = int(numtaps)
        self.cutoff  = float(cutoff)
        self.window  = window
        self.strides = (int(strides[0]), int(strides[1]))

    def build(self, input_shape):
        C = int(input_shape[-1])

        # Filtro 1D -> kernel 2D via produto externo
        fir_1d = scipy.signal.firwin(
            numtaps=self.numtaps,
            cutoff=self.cutoff,
            window=self.window,
        ).astype(np.float32)
        kernel_2d = np.outer(fir_1d, fir_1d)  # (numtaps, numtaps)

        # Replicar para todos os canais: (kh, kw, C, 1)
        kernel = np.stack([kernel_2d] * C, axis=2)[:, :, :, np.newaxis]

        self.fir_kernel = self.add_weight(
            name='fir_kernel',
            shape=kernel.shape,
            initializer=tf.keras.initializers.Constant(kernel),
            trainable=False,
        )

        s = self.strides
        self._pool = (
            tf.keras.layers.AveragePooling2D(pool_size=s, strides=s, padding='same')
            if s[0] > 1 or s[1] > 1 else None
        )
        super().build(input_shape)

    def call(self, x, training=None):
        out = tf.nn.depthwise_conv2d(
            x, self.fir_kernel,
            strides=[1, 1, 1, 1],
            padding='SAME',
        )
        return self._pool(out) if self._pool is not None else out

    def get_config(self):
        return {
            **super().get_config(),
            'numtaps': self.numtaps,
            'cutoff':  self.cutoff,
            'window':  self.window,
            'strides': self.strides,
        }

In [ ]:
# Coeficientes db4 hardcoded como fallback caso pywt não esteja instalado
_DB4_LO = np.array([
    -0.010597401784997278,  0.032883011666982945,  0.030841381835986965,
    -0.18703481171888114,   0.027983769416983849,  0.6308807679295904,
     0.7148465705525415,    0.23037781330885523,
], dtype=np.float32)


class DaubechiesDepthwise(tf.keras.layers.Layer):
    """
    Substitui DepthwiseConv2D pelo subband LL (passa-baixas 2D) da transformada
    wavelet Daubechies fixo (não treinável).

    kernel_2d = np.outer(dec_lo, dec_lo)   → (len(dec_lo), len(dec_lo))

    O mesmo kernel é aplicado a todos os canais (depthwise, channel_multiplier=1).
    O número de canais de saída é idêntico ao de entrada.

    Entrada : (B, H, W, C)
    Saída   : (B, H/sh, W/sw, C)
    """

    def __init__(self, wavelet=WAVELET_NAME, strides=(1, 1), **kwargs):
        super().__init__(**kwargs)
        self.wavelet_name = wavelet
        self.strides = (int(strides[0]), int(strides[1]))

    def build(self, input_shape):
        C = int(input_shape[-1])

        # Coeficientes lowpass do wavelet
        if _PYWT_AVAILABLE:
            dec_lo = np.array(pywt.Wavelet(self.wavelet_name).dec_lo, dtype=np.float32)
        else:
            if self.wavelet_name == 'db4':
                dec_lo = _DB4_LO
            else:
                raise ValueError(
                    f"pywt não disponível e '{self.wavelet_name}' não está hardcoded. "
                    "Instale pywt ou use wavelet='db4'."
                )

        # Kernel 2D LL: produto externo lowpass × lowpass
        kernel_2d = np.outer(dec_lo, dec_lo).astype(np.float32)  # (k, k)

        # Replicar para todos os canais: (kh, kw, C, 1)
        kernel = np.stack([kernel_2d] * C, axis=2)[:, :, :, np.newaxis]

        self.wavelet_kernel = self.add_weight(
            name='wavelet_kernel',
            shape=kernel.shape,
            initializer=tf.keras.initializers.Constant(kernel),
            trainable=False,
        )

        s = self.strides
        self._pool = (
            tf.keras.layers.AveragePooling2D(pool_size=s, strides=s, padding='same')
            if s[0] > 1 or s[1] > 1 else None
        )
        super().build(input_shape)

    def call(self, x, training=None):
        out = tf.nn.depthwise_conv2d(
            x, self.wavelet_kernel,
            strides=[1, 1, 1, 1],
            padding='SAME',
        )
        return self._pool(out) if self._pool is not None else out

    def get_config(self):
        return {
            **super().get_config(),
            'wavelet':  self.wavelet_name,
            'strides': self.strides,
        }


CUSTOM_OBJECTS = {
    'FIRDepthwise':       FIRDepthwise,
    'DaubechiesDepthwise': DaubechiesDepthwise,
}

## Funções Auxiliares

In [ ]:
def load_data(disp, img, folder=FOLDER_I):
    load = lambda fname: pickle.load(open(fname, 'rb'))
    X_tr = load(f'{folder}/X_{img}_train({disp}).pickle')
    y_tr = load(f'{folder}/y_train({disp}).pickle')
    X_va = load(f'{folder}/X_{img}_val({disp}).pickle')
    y_va = load(f'{folder}/y_val({disp}).pickle')
    X_te = load(f'{folder}/X_{img}_test({disp}).pickle')
    y_te = load(f'{folder}/y_test({disp}).pickle')
    return X_tr, y_tr, X_va, y_va, X_te, y_te


def model_size_mb(model):
    n_params = sum(tf.size(w).numpy() for w in model.weights)
    return n_params * 4 / (1024 ** 2)


def count_params(model):
    trainable     = sum(tf.size(w).numpy() for w in model.trainable_weights)
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    return trainable, non_trainable


def measure_latency_ms(model, x_sample, n_warmup=5, n_reps=20):
    """Latência mediana de inferência em ms (batch completo)."""
    for _ in range(n_warmup):
        _ = model(x_sample, training=False)
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        _ = model(x_sample, training=False)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))

## Construção do Modelo

In [ ]:
def build_feature_extractor(input_shape, filter_type='none', n_blocks=0, name=None):
    """
    Constrói extrator de features baseado em MobileNetV3Large.

    filter_type='none',  n_blocks=0 → baseline original, totalmente congelado.
    filter_type='fir',   n_blocks=N → primeiros N DepthwiseConv2D → FIRDepthwise.
    filter_type='wavelet', n_blocks=N → primeiros N DepthwiseConv2D → DaubechiesDepthwise.

    As camadas substituídas são não-treináveis. O restante da backbone fica
    congelado com os pesos ImageNet originais.
    """
    base = MobileNetV3Large(
        input_shape=input_shape, weights='imagenet', include_top=False,
    )

    if n_blocks == 0 or filter_type == 'none':
        for layer in base.layers:
            layer.trainable = False
        out = GlobalAveragePooling2D()(base.output)
        return Model(inputs=base.input, outputs=out,
                     name=name or 'FE_baseline')

    counter = [0]

    def clone_fn(layer):
        if isinstance(layer, DepthwiseConv2D):
            counter[0] += 1
            if counter[0] <= n_blocks:
                strides = layer.get_config().get('strides', (1, 1))
                layer_name = f'{filter_type}_{counter[0]}'
                if filter_type == 'fir':
                    return FIRDepthwise(strides=strides, name=layer_name)
                else:  # 'wavelet'
                    return DaubechiesDepthwise(strides=strides, name=layer_name)
        return layer

    cloned = tf.keras.models.clone_model(base, clone_function=clone_fn)

    # Copiar pesos ImageNet para as camadas não substituídas
    for base_layer in base.layers:
        try:
            cloned_layer = cloned.get_layer(base_layer.name)
            weights = base_layer.get_weights()
            if weights:
                cloned_layer.set_weights(weights)
        except (ValueError, Exception):
            pass  # camada substituída — sem pesos pré-treinados para copiar

    # Congelar tudo (FIR/wavelet são não-treináveis por design)
    for layer in cloned.layers:
        layer.trainable = False

    out = GlobalAveragePooling2D()(cloned.output)
    return Model(
        inputs=cloned.input, outputs=out,
        name=name or f'FE_{filter_type}_{n_blocks}blocks',
    )


def build_full_model(input_shape, filter_type='none', n_blocks=0):
    """Extrator de features + cabeça MLP idêntica à dos notebooks 4/5."""
    fe  = build_feature_extractor(input_shape, filter_type, n_blocks)
    x   = Dense(64, activation='relu')(fe.output)
    x   = Dropout(0.25)(x)
    x   = Dense(64, activation='relu')(x)
    x   = Dropout(0.25)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(
        inputs=fe.input, outputs=out,
        name=f'model_{filter_type}_{n_blocks}blocks',
    )
    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=1e-3),
        metrics=['accuracy'],
    )
    return model

## Verificação Rápida das Camadas (Smoke Test)

Confirma que as camadas customizadas produzem o shape correto e que o FE de cada variante resulta em `(None, 1280)` para as features.

In [ ]:
# Smoke test: usa um tensor sintético para verificar shapes
_dummy_shape = (224, 224, 3)  # shape típico dos dados
_x = tf.random.uniform((2, *_dummy_shape))

# Teste FIRDepthwise isolado
_fir = FIRDepthwise(strides=(1, 1))
_fir.build((None, *_dummy_shape))
assert _fir(_x).shape == _x.shape, 'FIRDepthwise: shape incorreto'
print('FIRDepthwise OK:', _fir(_x).shape)

# Teste FIRDepthwise com stride
_fir_s2 = FIRDepthwise(strides=(2, 2))
_fir_s2.build((None, *_dummy_shape))
assert _fir_s2(_x).shape == (2, 112, 112, 3)
print('FIRDepthwise stride=2 OK:', _fir_s2(_x).shape)

# Teste DaubechiesDepthwise isolado
_db4 = DaubechiesDepthwise(strides=(1, 1))
_db4.build((None, *_dummy_shape))
assert _db4(_x).shape == _x.shape, 'DaubechiesDepthwise: shape incorreto'
print('DaubechiesDepthwise OK:', _db4(_x).shape)

# Teste DaubechiesDepthwise com stride
_db4_s2 = DaubechiesDepthwise(strides=(2, 2))
_db4_s2.build((None, *_dummy_shape))
assert _db4_s2(_x).shape == (2, 112, 112, 3)
print('DaubechiesDepthwise stride=2 OK:', _db4_s2(_x).shape)

# Verificar output shape de cada variante do FE
print('\nVerificando output shape das FEs:')
for vname, (ftype, nblocks) in VARIANTS.items():
    fe = build_feature_extractor(_dummy_shape, ftype, nblocks)
    assert fe.output_shape == (None, 1280), (
        f'{vname}: output_shape esperado (None, 1280), obtido {fe.output_shape}'
    )
    print(f'  {vname}: {fe.output_shape}  ✓')
    K.clear_session()

print('\nTodos os smoke tests passaram!')

## Experimento Principal

In [ ]:
os.makedirs('output', exist_ok=True)
results = []

for variant_name, (filter_type, n_blocks) in VARIANTS.items():
    print(f"\n{'='*65}")
    print(f'VARIANTE: {variant_name}  (filter={filter_type}, n_blocks={n_blocks})')
    print(f"{'='*65}")

    for disp in DISPOSITIVOS:
        for img in IMAGENS:

            X_tr, y_tr, X_va, y_va, X_te, y_te = load_data(disp, img)
            input_shape = X_tr.shape[1:]

            run_accs, run_f1s, latencies = [], [], []
            size_mb = n_train = n_frozen = None

            for run in range(N_RUNS):
                ckpt = f'ckpt_fir_wavelet/{variant_name}/{disp}/{img}/run{run}/model.keras'
                os.makedirs(os.path.dirname(ckpt), exist_ok=True)

                model = build_full_model(input_shape, filter_type, n_blocks)

                if run == 0:
                    size_mb          = model_size_mb(model)
                    n_train, n_frozen = count_params(model)

                model.fit(
                    X_tr, y_tr,
                    batch_size=BATCH,
                    epochs=100,
                    verbose=0,
                    validation_data=(X_va, y_va),
                    callbacks=[
                        EarlyStopping(
                            monitor='val_loss', patience=7,
                            restore_best_weights=False, verbose=0,
                        ),
                        ModelCheckpoint(
                            ckpt, monitor='val_accuracy',
                            save_best_only=True, verbose=0,
                        ),
                    ],
                )

                best = tf.keras.models.load_model(ckpt, custom_objects=CUSTOM_OBJECTS)

                preds = (
                    best.predict(X_te, batch_size=BATCH, verbose=0) > 0.5
                ).astype(int).flatten()
                run_accs.append(float(np.mean(preds == y_te)))
                run_f1s.append(float(sklearn_f1(y_te, preds, average='macro')))

                # Latência medida no FE (somente extração de features, sem cabeça MLP)
                fe_model = Model(
                    inputs=best.input,
                    outputs=best.get_layer('global_average_pooling2d').output,
                )
                x_sample = X_te[:BATCH]
                latencies.append(measure_latency_ms(fe_model, x_sample))

                del model, best, fe_model
                K.clear_session()

            row = {
                'variant':           variant_name,
                'filter_type':       filter_type,
                'n_blocks':          n_blocks,
                'device':            disp,
                'image':             img,
                'acc_mean':          np.mean(run_accs),
                'acc_std':           np.std(run_accs),
                'f1_mean':           np.mean(run_f1s),
                'f1_std':            np.std(run_f1s),
                'latency_ms_mean':   np.mean(latencies),
                'latency_ms_std':    np.std(latencies),
                'model_size_mb':     size_mb,
                'trainable_params':  n_train,
                'frozen_params':     n_frozen,
            }
            results.append(row)

            print(
                f'  {disp:<16} | {img:<5} | '
                f'acc={np.mean(run_accs):.3f}±{np.std(run_accs):.3f} | '
                f'f1={np.mean(run_f1s):.3f} | '
                f'lat={np.mean(latencies):.1f}ms'
            )

df_results = pd.DataFrame(results)
df_results.to_csv('output/fir_wavelet_experiment_results.csv', index=False)
print('\nResultados salvos -> output/fir_wavelet_experiment_results.csv')
df_results

## Análise dos Resultados

In [ ]:
# Carregar resultados (caso o notebook seja reiniciado)
df_results = pd.read_csv('output/fir_wavelet_experiment_results.csv')

# Resumo por variante: média das métricas sobre todos os dispositivos e imagens
summary = df_results.groupby('variant').agg(
    acc_mean=('acc_mean', 'mean'),
    acc_std=('acc_std', 'mean'),
    f1_mean=('f1_mean', 'mean'),
    latency_ms=('latency_ms_mean', 'mean'),
    model_size_mb=('model_size_mb', 'first'),
    trainable_params=('trainable_params', 'first'),
    frozen_params=('frozen_params', 'first'),
).reset_index()

print('=== Resumo por Variante ===')
print(summary.to_string(index=False))

In [ ]:
from matplotlib import pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Acurácia média por variante
axes[0].barh(summary['variant'], summary['acc_mean'])
axes[0].set_xlabel('Acurácia Média')
axes[0].set_title('Acurácia (média sobre dispositivos e imagens)')
axes[0].set_xlim(0, 1)

# Latência média do FE por variante
axes[1].barh(summary['variant'], summary['latency_ms'])
axes[1].set_xlabel('Latência FE (ms)')
axes[1].set_title('Latência do Extrator de Features')

# Tamanho do modelo por variante
axes[2].barh(summary['variant'], summary['model_size_mb'])
axes[2].set_xlabel('Tamanho (MB)')
axes[2].set_title('Tamanho do Modelo')

plt.tight_layout()
plt.savefig('output/fir_wavelet_summary.png', dpi=150)
plt.show()